Maxwell Equations
===

We compute the magnetic field generated by a coil placed on a C-shaped iron core.

In [ ]:
from netgen.occ import *
from ngsolve import *
from ngsolve.webgui import Draw

Geometric model using the Netgen-OpenCascade interface. Core and coil are cut out of the air box, and `Glue` connects the three solids to one geometry with conforming interfaces:

In [ ]:
box = Box((-1,-1,-1), (2,1,2))
box.faces.name = "outer"

core = Box((0,-0.05,0), (0.8,0.05,1)) - \
       Box((0.1,-1,0.1), (0.7,1,0.9)) - \
       Box((0.5,-1,0.4), (1,1,0.6))
core.solids.name = "core"
core.solids.maxh = 0.2
core.faces.col = (0.5,0.5,0)

coil = Cylinder((0.05,0,0.3), Z, r=0.3, h=0.4) - \
       Cylinder((0.05,0,0.3), Z, r=0.15, h=0.4)
coil.solids.name = "coil"
coil.solids.maxh = 0.2
coil.faces.col = (0,1,0)

air = box - core - coil
air.solids.name = "air"

ea = { "euler_angles" : (-70,0,25) }
Draw (Glue([core,coil]), **ea);

In [ ]:
geo = Glue([core, coil, air])
mesh = geo.GenerateMesh(maxh=0.5).Curve(5)
Draw (mesh, clipping = { "pnt" : (0,0,0), "vec" : (0,1,0) }, **ea);

Magnetostatics:


find $u \in H(curl)$ such that
$$
\int \mu^{-1} \operatorname{curl} u \operatorname{curl} v = \int j v
$$
for all $v \in H(curl)$.

The permeability is given per material, the current density is prescribed in the coil. We solve by conjugate gradients with a BDDC preconditioner, `Solve` distributes the work by task-parallelization:

In [ ]:
fes = HCurl(mesh, order=4, nograds=True)
print ("ndof =", fes.ndof)
u,v = fes.TnT()

mu0 = 1.257e-6
mur = mesh.MaterialCF({ "core" : 1000 }, default=1)
nu = 1/(mu0*mur)
j = CF((y,0.05-x,0))

lhs = nu*curl(u)*curl(v)*dx + 1e-6*nu*u*v*dx
rhs = j*v*dx("coil")

gfu = Solve(lhs==rhs, u[BND("outer")]==(0,0,0),
            preconditioners.BDDC(),
            solvers.CGSolver.Creator(printrates=False))

In [ ]:
Draw (curl(gfu), mesh, "B-field", draw_surf=False, \
      clipping = { "pnt" : (0,0,0), "vec" : (0,1,0), "function" : False },
      vectors = { "grid_size" : 100 }, **ea);